In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Personal setting

In [ ]:
BASE_INPUT_PATH = "/kaggle/input/datasets/chiaracervelli/"
EMBEDDING_DIR = "embeddings" # Se hai creato una cartella apposita
TWEETS_DIR = "tweets"
BASE_WORKING_PATH = "/kaggle/working/"
PORT= 5500

## Install dependencies 

In [ ]:
!pip install umap-learn spacy nltk
!python -m spacy download en_core_web_sm
!pip install --upgrade transformers

In [ ]:
!pip install flask flask-cors pyngrok


## Imports

In [ ]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import umap
import spacy
from nltk.corpus import wordnet
import nltk
from tqdm import tqdm
import os
import joblib 
from sklearn.decomposition import PCA

# Download WordNet if not present
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient
# Tell pyngrok to use the cleanly installed apt version
conf.get_default().ngrok_path = "/usr/bin/ngrok"

# Safely retrieve the token from your Kaggle Secrets vault
user_secrets = UserSecretsClient()
my_secret_token = user_secrets.get_secret("NGROK_TOKEN")

## Model initialization

In [ ]:
# 1. Load Embedding Model (Qwen 0.6B)
emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading embedding model {emb_model_id}...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_tokenizer.padding_side = "right"
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

# 2. Load Generative Model (Gemma 4 E2B Instruct)
gen_model_id = "google/gemma-4-E2B-it"
print(f"Loading generative model {gen_model_id}...")

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="cuda:0"
)



## Embedding functions

In [ ]:
def get_embedding(text):
    """Calculates the embedding for a single text string."""
    inputs = emb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding

def batch_get_embeddings(texts, batch_size=32):
    """Calculates embeddings for a list of texts efficiently in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Calculating Batch Embeddings"):
        batch_texts = texts[i : i + batch_size]
        inputs = emb_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = emb_model(**inputs)
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

## Perturbation logic

In [ ]:

def get_wordnet_pos_from_spacy(spacy_pos):
    """Maps spaCy POS tags to WordNet formats."""
    if spacy_pos == 'ADJ': return wordnet.ADJ
    elif spacy_pos == 'VERB': return wordnet.VERB
    elif spacy_pos == 'NOUN': return wordnet.NOUN
    elif spacy_pos == 'ADV': return wordnet.ADV
    return None

def perturb_dict(text, mode="synonym"):
    """
    1 & 2. Dictionary Perturbation: Replaces a single word with a synonym or antonym.
    """
    doc = nlp(text)
    perturbed_text = ""
    changed = False

    for token in doc:
        if changed or not token.is_alpha:
            perturbed_text += token.text_with_ws
            continue

        wn_tag = get_wordnet_pos_from_spacy(token.pos_)
        if wn_tag is None:
            perturbed_text += token.text_with_ws
            continue

        synsets = wordnet.synsets(token.lemma_, pos=wn_tag)
        replacements = set()
        
        for syn in synsets:
            for lemma in syn.lemmas():
                if mode == "synonym" and lemma.name().lower() != token.lemma_.lower():
                    replacements.add(lemma.name().replace('_', ' '))
                elif mode == "antonym" and lemma.antonyms():
                    replacements.add(lemma.antonyms()[0].name().replace('_', ' '))

        if replacements:
            new_word = random.choice(list(replacements))
            if token.is_title: new_word = new_word.title()
            elif token.is_upper: new_word = new_word.upper()
            perturbed_text += new_word + token.whitespace_
            changed = True
        else:
            perturbed_text += token.text_with_ws

    return perturbed_text

def perturb_noise(text, noise_words=["blah", "xyz", "uhm", "err"]):
    """
    3. Noise Perturbation: Inserts random noise words into the tweet.
    """
    doc = nlp(text)
    if len(doc) == 0: return text
    
    tokens_with_ws = [token.text_with_ws for token in doc]
    num_noise = random.randint(1, 2)
    
    for _ in range(num_noise):
        insert_idx = random.randint(0, len(tokens_with_ws))
        noise = random.choice(noise_words) + " "
        tokens_with_ws.insert(insert_idx, noise)
        
    return "".join(tokens_with_ws)

def perturb_llm(text, mode="paraphrase"):
    """
    4 & 5. LLM Perturbation: Rephrases or flips sentiment completely.
    """
    if mode == "paraphrase":
        instruction = "Rewrite the following airline review tweet using completely different words, but keep the exact same meaning and sentiment."
    elif mode == "opposite":
        instruction = "Rewrite the following airline review tweet to have the exact opposite sentiment. If it complains, make it praise the airline. If it praises, make it complain."
        
    messages = [
        {
            "role": "system", 
            "content": "You are a helpful assistant specialized in rewriting text. Output ONLY the rewritten text, without explanations, conversational filler, markdown blocks, or quotes."
        },
        {
            "role": "user", 
            "content": f"{instruction}\n\nTweet: {text}"
        }
    ]
    
    text_input = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = gen_tokenizer([text_input], return_tensors="pt").to(gen_model.device)    
    with torch.no_grad():
        generated_ids = gen_model.generate(
            **model_inputs, 
            max_new_tokens=100, # Aumentato per evitare che Gemma tagli le risposte
            temperature=0.7,
            top_p=0.95,         # Raccomandato per Gemma
            top_k=64,           # Raccomandato per Gemma
            do_sample=True
        )
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    torch.cuda.empty_cache() # Svuota la VRAM dai calcoli temporanei non più necessari
    return gen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


## Visual analytics features

In [ ]:
global_umap_mapper = None

# 1. Scatterplot using UMAP
def generate_base_map_umap():
    global global_umap_mapper

    # Percorsi dei file. 
    csv_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "base_map.csv")
    model_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR,"umap_mapper.joblib")

    tweets_dataset_path = os.path.join(BASE_INPUT_PATH, TWEETS_DIR, "tweets.csv")
    
    # Check if files already exist 
    if os.path.exists(csv_path) and os.path.exists(model_path):
        print(f"Found existing '{csv_path}' and '{model_path}'.")
        print("Loading UMAP model into memory... (Skipping heavy embedding computation!)")
        global_umap_mapper = joblib.load(model_path)
        print("System ready.")
        return

    print("Files not found. Loading dataset to generate base map from scratch...")
    
    df_full = pd.read_csv(tweets_dataset_path, dtype={"tweet_id": str})
    df_sample = df_full.sample(n=10000, random_state=42).copy()
    
    texts = df_sample['text'].astype(str).tolist()
    
    # Use the batched function for speed
    embeddings_1024d = batch_get_embeddings(texts, batch_size=32)
    
    print("\nTraining UMAP for 2D projection...")
    
    # Inizializza la UMAP per 2 componenti
    umap_model = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        metric='cosine', 
        random_state=42
    )
    
    # Fit e transform per ottenere le coordinate 2D
    embeddings_2d = umap_model.fit_transform(embeddings_1024d.numpy())
    
    # Salviamo l'oggetto UMAP nella variabile globale
    global_umap_mapper = umap_model
    
    df_map = pd.DataFrame({
        "tweet_id": df_sample["tweet_id"],
        "text": df_sample["text"],
        "sentiment": df_sample["airline_sentiment"],
        "airline": df_sample["airline"],
        "x": embeddings_2d[:, 0], # x coordinates
        "y": embeddings_2d[:, 1]  # y coordinates
    })
    csv_save_path = os.path.join(BASE_WORKING_PATH, "base_map.csv")
    model_save_path = os.path.join(BASE_WORKING_PATH, "umap_mapper.joblib")
    
    df_map.to_csv(csv_save_path, index=False)
    print("Base map generated successfully and saved as 'base_map.csv'.")
    
    # Salva l'oggetto UMAP
    joblib.dump(global_umap_mapper, model_save_path)

# 2. Token weight analysis: take one tweet, and removes 1 token at a time, recomputes the embeddings of the new tweet
# without that token to measure the cosine similarity between the original embedding and the new one
def get_token_weights(tweet_text):
    orig_emb = get_embedding(tweet_text)
    tokens = emb_tokenizer.tokenize(tweet_text)
    results = []
    
    for i, token in enumerate(tokens):
        # Remove token to test structural importance
        # 2. Copiamo la lista e rimuoviamo SOLO il token in questa posizione esatta
        mod_tokens = tokens.copy()
        mod_tokens.pop(i) 
        
        # 3. Chiediamo al tokenizer di incollare di nuovo i pezzi in una stringa
        mod_text = emb_tokenizer.convert_tokens_to_string(mod_tokens)
        if not mod_text.strip(): 
            mod_text = "empty"
            
        mod_emb = get_embedding(mod_text)
        sim = F.cosine_similarity(orig_emb, mod_emb).item()
        
        # 1.0 - sim calculates the "drop" in similarity. Higher drop = higher weight.
        impact = 1.0 - sim 

        # 4. (Opzionale) Pulizia visiva. I token spesso hanno caratteri strani (es. 'Ġ' o ' '). 
        # Li rimuoviamo solo per non farli vedere brutti nel frontend D3.js.
        clean_visual_token = token.replace('Ġ', '').replace(' ', '').replace('##', '')
        
        results.append({
            "word": clean_visual_token if clean_visual_token else token,
            "weight": round(impact, 4)
        })
    return results

# 3. 
def generate_heatmap_data(tweet_text):
    """Generates the flattened matrix data for the D3 Heatmap."""
    tokens = emb_tokenizer.tokenize(tweet_text)
    
    # Take only the first 10 token 
    tokens_subset = tokens[:10]
    
    # Only token-level perturbations here for performance.
    perturbations = ["Remove", "Dict_Synonym", "Dict_Antonym"]
    
    heatmap_data = []
    orig_emb = get_embedding(tweet_text)
    
    for i, token in enumerate(tokens_subset):
        # Pulizia visiva per l'asse X della heatmap
        clean_visual_token = token.replace('Ġ', '').replace(' ', '').replace('##', '')
        if not clean_visual_token: clean_visual_token = token
            
        for pert in perturbations:
            if pert == "Remove":
                # Rimuoviamo esattamente questo token dalla lista completa
                mod_tokens = tokens.copy()
                mod_tokens.pop(i)
                mod_text = emb_tokenizer.convert_tokens_to_string(mod_tokens)
            
            # NOTA: Synonym e Antonym lavorano tipicamente su "parole" intere. 
            # Applicarli su "subwords" (es. "ing") potrebbe fallire.
            # Ma manteniamo la tua logica inviando la stringa originale ricostruita.
            elif pert == "Dict_Synonym":
                mod_text = perturb_dict(tweet_text, mode="synonym") 
            elif pert == "Dict_Antonym":
                mod_text = perturb_dict(tweet_text, mode="antonym")
                
            if not mod_text.strip(): mod_text = "empty"
            
            mod_emb = get_embedding(mod_text)
            sim = F.cosine_similarity(orig_emb, mod_emb).item()
            importance = 1.0 - sim
            
            heatmap_data.append({
                "token": clean_visual_token, # Usiamo il nome pulito per D3.js
                "perturbation": pert,
                "importance": round(importance, 4)
            })
            
    return heatmap_data

def get_perturbed_coordinates(tweet_text, perturbation_type="llm_paraphrase"):
    """
    Applies one of the 5 specific perturbations, embeds, and projects to UMAP.
    Valid perturbation_types: 'dict_synonym', 'dict_antonym', 'noise', 'llm_paraphrase', 'llm_opposite'
    """
    # Controllo di sicurezza per la UMAP
    if global_umap_mapper is None:
        raise ValueError("UMAP mapper not initialized. Run generate_base_map_umap() first.")
        
    # Route to the correct perturbation function
    if perturbation_type == "dict_synonym":
        new_text = perturb_dict(tweet_text, mode="synonym")
    elif perturbation_type == "dict_antonym":
        new_text = perturb_dict(tweet_text, mode="antonym")
    elif perturbation_type == "noise":
        new_text = perturb_noise(tweet_text)
    elif perturbation_type == "llm_paraphrase":
        new_text = perturb_llm(tweet_text, mode="paraphrase")
    elif perturbation_type == "llm_opposite":
        new_text = perturb_llm(tweet_text, mode="opposite")
    else:
        new_text = tweet_text
        
    new_emb_tensor = get_embedding(new_text)
    new_emb_numpy = new_emb_tensor.cpu().numpy()
    
    # Project into existing 2D space
    new_2d_coords = global_umap_mapper.transform(new_emb_numpy)
    
    return {
        "original_text": tweet_text,
        "new_text": new_text,
        "perturbation_applied": perturbation_type,
        "new_x": float(new_2d_coords[0][0]),
        "new_y": float(new_2d_coords[0][1])
    }

## Main logic

In [ ]:
import warnings
import os
import subprocess
warnings.filterwarnings('ignore')

if __name__ == "__main__":
    # 1. Generate the Base Map and train the UMAP model in memory
    print("--- STEP 1: INITIALIZING SYSTEM ---")
    generate_base_map_umap()
    
    # 2. Set up the Flask API Server
    print("\n--- STEP 2: STARTING API SERVER ---")
    
    # We install the OFFICIAL ngrok library (which bypasses the 403 error entirely)
    print("Installing required server packages...")
    subprocess.run(["pip", "install", "-q", "ngrok", "flask", "flask-cors"])
    
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import ngrok
    from kaggle_secrets import UserSecretsClient

    app = Flask(__name__)
    CORS(app)

    @app.route('/api/perturb', methods=['POST'])
    def perturb_endpoint():
        data = request.json
        tweet_text = data.get('text')
        perturb_type = data.get('type', 'llm_paraphrase') 
        try:
            result = get_perturbed_coordinates(tweet_text, perturb_type)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/token_weights', methods=['POST'])
    def token_weights_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = get_token_weights(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/heatmap', methods=['POST'])
    def heatmap_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = generate_heatmap_data(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    # --- KAGGLE NGROK SETUP (OFFICIAL SDK METHOD) ---
    print("Configuring ngrok...")
    try:
        user_secrets = UserSecretsClient()
        my_secret_token = user_secrets.get_secret("NGROK_TOKEN")
        
        listener = await ngrok.forward(5000, authtoken=my_secret_token)
        public_url = listener.url()
        
        print(f"\n✅ YOUR API IS LIVE AT: {public_url}")
        print("Copy this URL into your app.js fetch() calls!")
        
        # Avviamo il server nel blocco try
        print("\n⏳ Server in ascolto... Premi il tasto 'Stop' (⏹️) accanto alla cella per spegnerlo in modo sicuro.")
        app.run(port=PORT)

    except KeyboardInterrupt:
        # QUESTO SCATTA QUANDO PREMI IL TASTO STOP SU KAGGLE
        print("\n🛑 Interruzione rilevata (Ctrl+C). Chiusura del tunnel in corso...")
        listener.close() # Chiude il tunnel Ngrok liberando l'account!
        print("✅ Tunnel Ngrok chiuso e server spento in modo pulito.")
        
    except Exception as e:
        print("\n⚠️ Errore imprevisto:")
        raise e